# Modul 3: Operasi Citra Sederhana

Nama: Sahrul Ramadhani  
NIM: isi NIM Anda  
Kelas: isi kelas Anda

## D1. Operasi Citra Sederhana

### 1. Membuka notebook di Google Colab

Nama notebook: `Week3.ipynb`

### 2. Menghubungkan Google Colab dengan Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Folder gambar yang dipakai: `MyDrive/PCVK/Minggu 3/images`.

In [ ]:
import os

folder_drive = '/content/drive/MyDrive/PCVK/Minggu 3/images'
os.chdir(folder_drive)
print('Folder aktif:', os.getcwd())

In [ ]:
import cv2 as cv
from google.colab.patches import cv2_imshow
import numpy as np
import matplotlib.pyplot as plt
import glob
from math import log10, sqrt
from pathlib import Path
import pandas as pd

file_wajib = ['houses.jpg', 'peppers.jpg', 'galaxy.jpg', 'couple.tiff', 'night.jpg', 'crayfish.jpg']
file_hilang = [nama for nama in file_wajib if not Path(nama).exists()]
jumlah_noise = len(glob.glob('noises/*.jpg'))

if file_hilang:
    raise FileNotFoundError('File belum tersedia: ' + ', '.join(file_hilang))

if jumlah_noise < 100:
    raise FileNotFoundError(f'Noise hanya {jumlah_noise}. Siapkan 100 file JPG di folder noises.')

print('Semua file utama tersedia.')
print('Jumlah citra noise:', jumlah_noise)

### 3. Transformasi Linier Brightness

Formula: `g(x,y) = f(x,y) + b`

In [ ]:
print(' Mengubah tingkat kecerahan citra ')
print('----------------------------------')

brightness = 30
try:
    brightness = int(input('Masukkan nilai kecerahan: '))
except ValueError:
    print('Error, not a number. Nilai 30 digunakan.')

original = cv.imread('houses.jpg')
brightness_image = np.zeros(original.shape, original.dtype)

for y in range(original.shape[0]):
    for x in range(original.shape[1]):
        for c in range(original.shape[2]):
            brightness_image[y,x,c] = np.clip(int(original[y,x,c]) + brightness, 0, 255)

final_frame = cv.hconcat((original, brightness_image))
cv2_imshow(final_frame)

## Tugas Praktikum

### 1. Inverse citra

Formula: `g(x,y) = 255 - f(x,y)`

In [ ]:
original = cv.imread('peppers.jpg')
inverse_image = 255 - original

final_frame = cv.hconcat((original, inverse_image))
cv2_imshow(final_frame)

### 2. Transformasi contrast

Formula: `g(x,y) = a * f(x,y) + b`

In [ ]:
print(' Mengubah kontras dan tingkat kecerahan citra ')
print('----------------------------------------------')

brightness = 10
contrast = 1.5

try:
    brightness = int(input('Masukkan tingkat kecerahan: '))
    contrast = float(input('Masukkan kontras: '))
except ValueError:
    print('Error, not a number. Nilai 10 dan 1.5 digunakan.')

original = cv.imread('houses.jpg')
contrast_image = np.clip(contrast * original.astype(np.float32) + brightness, 0, 255).astype(np.uint8)

final_frame = cv.hconcat((original, contrast_image))
cv2_imshow(final_frame)

### 3. Transformasi logarithmic brightness

Formula: `s = c * log(1 + r)`

In [ ]:
print(' Mengubah tingkat kecerahan citra dengan Transformasi Log ')
print('----------------------------------------------------------')

c = 50.0
try:
    c = float(input('Masukkan nilai kecerahan: '))
except ValueError:
    print('Error, not a number. Nilai 50 digunakan.')

original = cv.imread('houses.jpg')
log_image = np.clip(c * np.log1p(original.astype(np.float32)), 0, 255).astype(np.uint8)

final_frame = cv.hconcat((original, log_image))
cv2_imshow(final_frame)

### 4. Grayscale averaging, lightness, dan luminance

In [ ]:
original = cv.imread('peppers.jpg')

B = original[:,:,0].astype(np.float32)
G = original[:,:,1].astype(np.float32)
R = original[:,:,2].astype(np.float32)

gray_averaging = ((R + G + B) / 3).astype(np.uint8)
gray_lightness = ((np.maximum(np.maximum(R,G),B) + np.minimum(np.minimum(R,G),B)) / 2).astype(np.uint8)
gray_luminance = (0.21 * R + 0.72 * G + 0.07 * B).astype(np.uint8)

plt.figure(figsize=(12,9))

plt.subplot(2,2,1)
plt.imshow(cv.cvtColor(original, cv.COLOR_BGR2RGB))
plt.title('Citra Asli')
plt.axis('off')

plt.subplot(2,2,2)
plt.imshow(gray_averaging, cmap='gray')
plt.title('Averaging')
plt.axis('off')

plt.subplot(2,2,3)
plt.imshow(gray_lightness, cmap='gray')
plt.title('Lightness')
plt.axis('off')

plt.subplot(2,2,4)
plt.imshow(gray_luminance, cmap='gray')
plt.title('Luminance')
plt.axis('off')

plt.tight_layout()
plt.show()

### 5. Menampilkan warna merah dan mengubah warna lain menjadi grayscale

In [ ]:
original = cv.imread('peppers.jpg')
hsv = cv.cvtColor(original, cv.COLOR_BGR2HSV)

mask_merah_1 = cv.inRange(hsv, np.array([0,70,50]), np.array([10,255,255]))
mask_merah_2 = cv.inRange(hsv, np.array([170,70,50]), np.array([180,255,255]))
mask_merah = cv.bitwise_or(mask_merah_1, mask_merah_2)

gray = cv.cvtColor(original, cv.COLOR_BGR2GRAY)
gray_bgr = cv.cvtColor(gray, cv.COLOR_GRAY2BGR)
hasil_merah = np.where(mask_merah[:,:,None] > 0, original, gray_bgr)

final_frame = cv.hconcat((original, hasil_merah))
cv2_imshow(final_frame)

### 6. Gamma Correction

Formula: `I' = 255 * (I / 255) ** (1 / gamma)`

In [ ]:
print(' Gamma Correction pada citra ')
print('----------------------------------')

gamma = 3
try:
    gamma = int(input('Masukkan nilai Gamma: '))
except ValueError:
    print('Error, not a number. Nilai 3 digunakan.')

if gamma <= 0:
    gamma = 3.0
    print('Gamma harus lebih besar dari 0. Nilai 3 digunakan.')

original = cv.imread('houses.jpg')
table = np.array([255 * ((i / 255) ** (1 / gamma)) for i in np.arange(256)]).astype(np.uint8)
gamma_image = cv.LUT(original, table)

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.imshow(cv.cvtColor(original, cv.COLOR_BGR2RGB))
plt.title('Citra Asli')
plt.axis('off')

plt.subplot(1,2,2)
plt.imshow(cv.cvtColor(gamma_image, cv.COLOR_BGR2RGB))
plt.title(f'Gamma Correction (gamma = {gamma:g})')
plt.axis('off')

plt.tight_layout()
plt.show()

### 7. Simulasi Image Depth

In [ ]:
bit_depth = 3
try:
    bit_depth = int(input('Masukkan bit depth tujuan (1-8): '))
except ValueError:
    print('Error, not a number. Nilai 3 digunakan.')

bit_depth = int(np.clip(bit_depth, 1, 8))
level = 255 / (pow(2,bit_depth) - 1)
original = cv.imread('peppers.jpg', cv.IMREAD_GRAYSCALE)
depth_image = np.round(original.astype(np.float32) / level) * level
depth_image = np.clip(depth_image, 0, 255).astype(np.uint8)

print('Bit depth awal  : 8 bit')
print('Bit depth tujuan:', bit_depth, 'bit')
print('Jumlah level    :', pow(2,bit_depth))

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.imshow(original, cmap='gray', vmin=0, vmax=255)
plt.title('Grayscale 8-bit')
plt.axis('off')

plt.subplot(1,2,2)
plt.imshow(depth_image, cmap='gray', vmin=0, vmax=255)
plt.title(f'Grayscale {bit_depth}-bit')
plt.axis('off')

plt.tight_layout()
plt.show()

### 8. Average Denoising

Citra asli: `galaxy.jpg`  
Seratus citra noise: `noises/*.jpg`  
Gunakan 100 citra noise asli yang diberikan dosen agar nilai PSNR sama dengan data praktikum.

In [ ]:
def PSNR(img1, img2):
    data1 = img1.astype(np.float64)
    data2 = img2.astype(np.float64)
    mse = np.mean((data1 - data2) ** 2)
    if mse == 0:
        return 100.0
    max_pixel = 255.0
    return 20 * log10(max_pixel / sqrt(mse))


original = cv.imread('galaxy.jpg')
daftar_file = sorted(glob.glob('noises/*.jpg'))
cv_img = []

for nama_file in daftar_file:
    gambar = cv.imread(nama_file)
    if gambar is not None and gambar.shape == original.shape:
        cv_img.append(gambar)

jumlah_citra = [10,20,40,80,100]
hasil_average = []
nilai_psnr = []

for jumlah in jumlah_citra:
    average_image = np.mean(np.stack(cv_img[:jumlah]).astype(np.float32), axis=0)
    average_image = np.clip(average_image, 0, 255).astype(np.uint8)
    hasil_average.append(average_image)
    nilai_psnr.append(PSNR(original, average_image))

tabel_psnr = pd.DataFrame({
    'Jumlah Citra di Average': jumlah_citra,
    'Nilai PSNR (dB)': [round(nilai, 2) for nilai in nilai_psnr]
})

display(tabel_psnr)

plt.figure(figsize=(15,8))
for indeks, (jumlah, gambar, nilai) in enumerate(zip(jumlah_citra, hasil_average, nilai_psnr), start=1):
    plt.subplot(2,3,indeks)
    plt.imshow(cv.cvtColor(gambar, cv.COLOR_BGR2RGB))
    plt.title(f'Average {jumlah} citra\nPSNR {nilai:.2f} dB')
    plt.axis('off')

plt.subplot(2,3,6)
plt.imshow(cv.cvtColor(original, cv.COLOR_BGR2RGB))
plt.title('Citra Asli')
plt.axis('off')

plt.tight_layout()
plt.show()

kenaikan = np.diff(nilai_psnr)
indeks_terkecil = int(np.argmin(kenaikan))
print(f'Peningkatan paling kecil terjadi dari {jumlah_citra[indeks_terkecil]} ke {jumlah_citra[indeks_terkecil+1]} citra, yaitu {kenaikan[indeks_terkecil]:.2f} dB.')

Tidak, kenaikan PSNR tidak selalu besar. Peningkatan paling kecil terjadi dari 80 ke 100 citra. Kesimpulannya, semakin banyak citra yang dirata-rata, noise berkurang dan PSNR naik, tetapi kenaikannya makin kecil saat jumlah citra sudah banyak.

### 9. Image Masking

In [ ]:
original = cv.imread('couple.tiff')
h, w = original.shape[:2]

mask = np.zeros((h,w), dtype=np.uint8)
radius = min(h,w) // 6
cv.circle(mask, (w//3,h//4), radius, 255, -1)
cv.circle(mask, (2*w//3,h//4), radius, 255, -1)

hasil_masking = cv.bitwise_and(original, original, mask=mask)

plt.figure(figsize=(13,4))

plt.subplot(1,3,1)
plt.imshow(cv.cvtColor(original, cv.COLOR_BGR2RGB))
plt.title('Citra Asli')
plt.axis('off')

plt.subplot(1,3,2)
plt.imshow(mask, cmap='gray')
plt.title('Mask')
plt.axis('off')

plt.subplot(1,3,3)
plt.imshow(cv.cvtColor(hasil_masking, cv.COLOR_BGR2RGB))
plt.title('Hasil AND dengan Mask')
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
gray = cv.cvtColor(original, cv.COLOR_BGR2GRAY)

hasil_not = cv.bitwise_not(gray)
hasil_or = cv.bitwise_or(gray, mask)
hasil_and = cv.bitwise_and(gray, mask)
hasil_nand = cv.bitwise_not(hasil_and)
hasil_xor = cv.bitwise_xor(gray, mask)

nama_operator = ['NOT', 'OR', 'AND', 'NAND', 'XOR']
hasil_operator = [hasil_not, hasil_or, hasil_and, hasil_nand, hasil_xor]

plt.figure(figsize=(15,8))

plt.subplot(2,3,1)
plt.imshow(gray, cmap='gray')
plt.title('Image Input')
plt.axis('off')

for indeks, (nama, hasil) in enumerate(zip(nama_operator, hasil_operator), start=2):
    plt.subplot(2,3,indeks)
    plt.imshow(hasil, cmap='gray', vmin=0, vmax=255)
    plt.title(nama)
    plt.axis('off')

plt.tight_layout()
plt.show()

NOT membalik nilai terang dan gelap pada citra. OR membuat bagian mask menjadi putih. AND hanya mempertahankan bagian yang dipilih mask. NAND adalah kebalikan dari AND. XOR menampilkan bagian yang berbeda antara citra dan mask.

### 10. Foto Malam Hari

Metode yang dipakai adalah Gamma Correction dengan gamma 2. Nilai ini menaikkan detail pada bagian gelap tanpa menambah nilai yang sama pada semua piksel. Risikonya adalah noise pada area gelap ikut terlihat dan bagian terang dapat kehilangan detail jika gamma terlalu besar.

Foto: Paul Harrop, [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Deep_dark_night,_Hull_-_geograph.org.uk_-_2281688.jpg), CC BY-SA 2.0.

In [ ]:
original = cv.imread('night.jpg')
gamma_malam = 2.0
table = np.array([255 * ((i / 255) ** (1 / gamma_malam)) for i in np.arange(256)]).astype(np.uint8)
hasil_malam = cv.LUT(original, table)

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.imshow(cv.cvtColor(original, cv.COLOR_BGR2RGB))
plt.title('Foto Malam Asli')
plt.axis('off')

plt.subplot(1,2,2)
plt.imshow(cv.cvtColor(hasil_malam, cv.COLOR_BGR2RGB))
plt.title('Gamma Correction (gamma = 2)')
plt.axis('off')

plt.tight_layout()
plt.show()

### 11. Perbaikan citra `crayfish.jpg`

Metode yang dipakai adalah contrast stretching. Batas bawah memakai persentil 2 dan batas atas memakai persentil 98. Nilai tersebut dipilih agar kabut abu-abu berkurang dan bentuk crayfish lebih jelas tanpa membuang terlalu banyak piksel ekstrem. Risikonya adalah noise dan warna kasar ikut diperkuat.

In [ ]:
original = cv.imread('crayfish.jpg')

batas_bawah = float(np.percentile(original, 2))
batas_atas = float(np.percentile(original, 98))
contrast = 255 / (batas_atas - batas_bawah)
brightness = -contrast * batas_bawah

hasil_crayfish = np.clip(contrast * original.astype(np.float32) + brightness, 0, 255).astype(np.uint8)

print(f'Batas bawah: {batas_bawah:.2f}')
print(f'Batas atas : {batas_atas:.2f}')
print(f'Contrast   : {contrast:.2f}')
print(f'Brightness : {brightness:.2f}')

plt.figure(figsize=(14,5))

plt.subplot(1,2,1)
plt.imshow(cv.cvtColor(original, cv.COLOR_BGR2RGB))
plt.title('Before')
plt.axis('off')

plt.subplot(1,2,2)
plt.imshow(cv.cvtColor(hasil_crayfish, cv.COLOR_BGR2RGB))
plt.title('After: Contrast Stretching')
plt.axis('off')

plt.tight_layout()
plt.show()